In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/data5/zhangye/CORAL/output/PBMC/single_000/T3_eval_cells.csv")

print(df["CORAL_time"].describe())
print(df.groupby("cell_type")["CORAL_time"].agg(["count", "mean", "std", "min", "max"]).sort_values("mean"))

count    2382.000000
mean        0.093910
std         0.142620
min         0.000000
25%         0.037882
50%         0.053741
75%         0.084120
max         1.000000
Name: CORAL_time, dtype: float64
                   count      mean       std       min       max
cell_type                                                       
MAIT                   3  0.025547  0.010425  0.016136  0.036752
HSPC                   5  0.026739  0.007582  0.020390  0.039919
CD8 Naive             66  0.033261  0.021300  0.018708  0.193357
CD4 Naive             31  0.036028  0.014907  0.020571  0.101735
CD4 Proliferating      1  0.036068       NaN  0.036068  0.036068
NK Proliferating       1  0.038637       NaN  0.038637  0.038637
CD8 TEM               21  0.038711  0.016142  0.020166  0.069550
NK                    13  0.054836  0.016235  0.018616  0.075593
CD4 TCM              150  0.056995  0.040911  0.016960  0.195310
CD14 Mono            104  0.057586  0.064409  0.016193  0.434589
Plasmablast        

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/data5/zhangye/CORAL/output/PBMC/single_000/T3_eval_cells.csv")

stage_order = {
    "HSPC":0,
    "B naive":1,"CD4 Naive":1,"CD8 Naive":1,"Eryth":1,
    "B intermediate":2,"CD4 TCM":2,"CD8 TCM":2,"CD14 Mono":2,"CD16 Mono":2,
    "cDC1":2,"cDC2":2,"pDC":2,"CD4 Proliferating":2,"NK Proliferating":2,
    "B memory":3,"Plasmablast":3,"CD4 TEM":3,"CD8 TEM":3,"Treg":3,
    "MAIT":3,"NK":3,"dnT":3,"gdT":3
}

df["stage"] = df["cell_type"].map(stage_order)

print(df.groupby("stage")["CORAL_time"].agg(["count", "mean", "std", "min", "max"]))
print(df.groupby("stage")["CORAL_time"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

       count      mean       std       min       max
stage                                               
0          5  0.026739  0.007582  0.020390  0.039919
1       1911  0.096622  0.147804  0.000000  1.000000
2        334  0.091735  0.137555  0.016193  0.971764
3        132  0.062685  0.042504  0.016136  0.248452
stage      
0      0.10    0.021875
       0.25    0.024104
       0.50    0.024305
       0.75    0.024980
       0.90    0.033943
1      0.10    0.030003
       0.25    0.039274
       0.50    0.056976
       0.75    0.084407
       0.90    0.150201
2      0.10    0.025404
       0.25    0.031177
       0.50    0.045311
       0.75    0.089375
       0.90    0.183772
3      0.10    0.026029
       0.25    0.032522
       0.50    0.049090
       0.75    0.070319
       0.90    0.129352
Name: CORAL_time, dtype: float64


In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

def per_stage_pair_auc(df, time_key="CORAL_time", stage_key="stage"):
    pred = pd.to_numeric(df[time_key], errors="coerce").to_numpy()
    ref = pd.to_numeric(df[stage_key], errors="coerce").to_numpy()

    valid = np.isfinite(pred) & np.isfinite(ref)
    pred = pred[valid]
    ref = ref[valid]

    rows = []
    classes = np.sort(np.unique(ref))

    for i, a in enumerate(classes):
        for b in classes[i + 1:]:
            mask = (ref == a) | (ref == b)
            y = (ref[mask] == b).astype(int)  # higher stage is positive
            s = pred[mask]

            if len(np.unique(y)) < 2:
                continue

            auc = roc_auc_score(y, s)
            n_a = int(np.sum(ref == a))
            n_b = int(np.sum(ref == b))

            rows.append({
                "stage_pair": f"{int(a)}->{int(b)}",
                "stage_low": int(a),
                "stage_high": int(b),
                "n_low": n_a,
                "n_high": n_b,
                "n_pair": n_a * n_b,
                "auc": float(auc),
                "auc_flipped": float(1.0 - auc),
                "mean_low": float(np.mean(pred[ref == a])),
                "mean_high": float(np.mean(pred[ref == b])),
                "direction_ok": bool(np.mean(pred[ref == b]) > np.mean(pred[ref == a])),
            })

    return pd.DataFrame(rows)

# df = pd.read_csv("/data5/zhangye/CORAL/output/PBMC/single_000/T3_eval_cells.csv")

# # 如果文件里还没有 stage，就先 map 一下
# stage_order = {
#     "HSPC":0,
#     "B naive":1,"CD4 Naive":1,"CD8 Naive":1,"Eryth":1,
#     "B intermediate":2,"CD4 TCM":2,"CD8 TCM":2,"CD14 Mono":2,"CD16 Mono":2,
#     "cDC1":2,"cDC2":2,"pDC":2,"CD4 Proliferating":2,"NK Proliferating":2,
#     "B memory":3,"Plasmablast":3,"CD4 TEM":3,"CD8 TEM":3,"Treg":3,
#     "MAIT":3,"NK":3,"dnT":3,"gdT":3
# }
# df["stage"] = df["cell_type"].map(stage_order)

# res = per_stage_pair_auc(df, time_key="CORAL_time", stage_key="stage")
# print(res)
# print("weighted_POAUC_like =", np.average(res["auc"], weights=res["stage_high"] - res["stage_low"]))

In [5]:
for r in ["single_000", "single_020", "single_060"]:
    path = f"/data5/zhangye/CORAL/output/PBMC_dynamic_unsup/{r}/T3_eval_cells.csv"
    df = pd.read_csv(path)
    df["stage"] = df["cell_type"].map(stage_order)

    a = per_stage_pair_auc(df, "CORAL_time", "stage")
    auc = np.average(a["auc"], weights=a["stage_high"] - a["stage_low"])

    df["CORAL_time_flip"] = 1.0 - df["CORAL_time"]
    b = per_stage_pair_auc(df, "CORAL_time_flip", "stage")
    auc_flip = np.average(b["auc"], weights=b["stage_high"] - b["stage_low"])

    print(r, "auc=", auc, "auc_flip=", auc_flip)

single_000 auc= 0.7632211562793255 auc_flip= 0.23677884372067431
single_020 auc= 0.521032320652494 auc_flip= 0.47896767934750584
single_060 auc= 0.545746576494204 auc_flip= 0.45425342350579606
